In [20]:
import json
import os
from pathlib import Path
from typing import Dict, List, Set, Tuple, Optional
from collections import defaultdict

# Load query rules
from query_rules import QUERY_RULES


In [21]:
def load_timeline_data(mix_dir: Path, domain: str, directory_index: str) -> Tuple[List[Dict], Dict, List[Dict]]:
    """
    Load events, artifacts, and concepts for a given timeline.
    
    Returns:
        (events, artifact_history, concepts)
        - events: List of event dicts
        - artifact_history: Dict mapping artifact_label -> List of artifact versions (each with event_id)
        - concepts: List of concept dicts (each with event_id)
    """
    timeline_dir = mix_dir / domain / directory_index
    
    # Load events
    events_file = timeline_dir / "events.json"
    with open(events_file, 'r', encoding='utf-8') as f:
        events = json.load(f)
    
    # Load artifacts (from artifact_store.json)
    artifact_store_file = timeline_dir / "artifacts" / "artifact_store.json"
    artifact_history = {}
    if artifact_store_file.exists():
        with open(artifact_store_file, 'r', encoding='utf-8') as f:
            artifact_store = json.load(f)
            # Extract history for each artifact
            if "history" in artifact_store:
                artifact_history = artifact_store["history"]
    
    # Load concepts
    concepts_file = timeline_dir / "concept" / "concepts.json"
    concepts = []
    if concepts_file.exists():
        with open(concepts_file, 'r', encoding='utf-8') as f:
            concepts = json.load(f)
    
    return events, artifact_history, concepts


In [22]:
def get_available_artifacts_up_to_event(events: List[Dict], event_idx: int) -> Set[str]:
    """
    Get all artifacts that have been generated up to and including event_idx.
    """
    available = set()
    for i in range(event_idx + 1):
        if "generated_artifacts" in events[i]:
            available.update(events[i]["generated_artifacts"])
    return available


def get_artifact_versions_up_to_event(artifact_history: Dict, artifact_label: str, event_idx: int, events: List[Dict]) -> List[str]:
    """
    Get all event_ids where the artifact was created/updated up to event_idx.
    Returns list of event_ids.
    """
    versions = []
    if artifact_label not in artifact_history:
        return versions
    
    # Get event_id to time_index mapping
    event_id_to_idx = {event["event_id"]: i for i, event in enumerate(events)}
    
    for artifact_version in artifact_history[artifact_label]:
        if "event_id" in artifact_version:
            event_id = artifact_version["event_id"]
            if event_id in event_id_to_idx and event_id_to_idx[event_id] <= event_idx:
                versions.append(event_id)
    
    return versions


def check_artifacts_changed(artifact_history: Dict, essentials: List[str], 
                            event_idx1: int, event_idx2: int, events: List[Dict]) -> bool:
    """
    Check if essentials artifacts changed between event_idx1 and event_idx2.
    
    For 3 artifacts: at least 1 must change
    For 4 artifacts: at least 2 must change
    """
    if event_idx1 >= event_idx2:
        return False
    
    num_essentials = len(essentials)
    required_changes = 1 if num_essentials == 3 else 2 if num_essentials >= 4 else 1
    
    changes_count = 0
    for artifact_label in essentials:
        versions1 = get_artifact_versions_up_to_event(artifact_history, artifact_label, event_idx1, events)
        versions2 = get_artifact_versions_up_to_event(artifact_history, artifact_label, event_idx2, events)
        
        # Check if there are new versions between event_idx1 and event_idx2
        new_versions = [v for v in versions2 if v not in versions1]
        if new_versions:
            changes_count += 1
    
    return changes_count >= required_changes


In [23]:
def insert_queries_for_timeline(events: List[Dict], artifact_history: Dict, concepts: List[Dict],
                                 domain: str, query_rules: Dict) -> List[Dict]:
    """
    Add query information to events.
    
    Returns:
        Events list with query field added to each event.
    """
    # Initialize query list for each event
    event_queries = [[] for _ in range(len(events))]
    
    # 1. Handle Concept Explanation tasks
    concept_explanation_config = query_rules.get("Concept Explanation", {})
    concept_explanation_query_template = concept_explanation_config.get("query", "")
    concept_explanation_essentials = concept_explanation_config.get("essentials", [])
    
    for concept in concepts:
        if "event_id" not in concept:
            continue
        
        # Find the event index for this concept's event_id
        concept_event_idx = None
        for i, event in enumerate(events):
            if event["event_id"] == concept["event_id"]:
                concept_event_idx = i
                break
        
        if concept_event_idx is None:
            continue
        
        # Skip first 3 events
        if concept_event_idx < 3:
            continue
        
        # Check if all essentials are available up to this event
        if concept_explanation_essentials:
            available_artifacts = get_available_artifacts_up_to_event(events, concept_event_idx)
            if not all(art in available_artifacts for art in concept_explanation_essentials):
                continue  # Skip this concept if essentials are not available
        
        # Create query for this concept
        concept_name = concept.get("concept", "[concept]")
        query_text = concept_explanation_query_template.replace("[concept]", concept_name)
        
        query_info = {
            "task": "Concept Explanation",
            "query": query_text,
            "target": concept_name,  # For concept explanation, target is the concept name
            "essential": concept_explanation_essentials.copy()  # Use essentials from query_rules
        }
        event_queries[concept_event_idx].append(query_info)
    
    # 2. Handle non-Concept Explanation tasks
    for task_type, task_config in query_rules.items():
        if task_type == "Concept Explanation":
            continue
        
        # Iterate through all subtasks (e.g., research_plan, method_scheme for "Plan & Design")
        for subtask_key, subtask_info in task_config.items():
            if not isinstance(subtask_info, dict) or "query" not in subtask_info:
                continue
            
            query_text = subtask_info["query"]
            essentials = subtask_info.get("essentials", [])
            targets = subtask_info.get("targets", [])
            
            # Find all valid insertion points
            valid_insertions = []
            for i in range(len(events)):
                # Skip first 3 events
                if i < 3:
                    continue
                # Check if all essentials are available up to this event
                available_artifacts = get_available_artifacts_up_to_event(events, i)
                if all(art in available_artifacts for art in essentials):
                    valid_insertions.append(i)
            
            # Select insertion points with diversity constraints
            selected_insertions = []
            for insertion_idx in valid_insertions:
                # Check interval constraint: at least 3 events from last insertion
                if selected_insertions:
                    last_insertion = selected_insertions[-1]
                    if insertion_idx - last_insertion < 5:
                        continue
                    
                    # Check change constraint: essentials must have changed
                    if not check_artifacts_changed(artifact_history, essentials, 
                                                  last_insertion, insertion_idx, events):
                        continue
                
                selected_insertions.append(insertion_idx)
            
            # Create query info for selected insertions
            for insertion_idx in selected_insertions:
                query_info = {
                    "task": task_type,
                    "query": query_text,
                    "target": targets[0] if targets else subtask_key,  # Use first target or subtask_key as target
                    "essential": essentials.copy()
                }
                event_queries[insertion_idx].append(query_info)
    
    # 3. Add query field to each event
    result_events = []
    for i, event in enumerate(events):
        # Create a copy of the event to avoid modifying the original
        new_event = event.copy()
        new_event["query"] = event_queries[i]
        result_events.append(new_event)
    
    return result_events


In [24]:
def process_all_timelines(mix_dir: Path, output_dir: Path, domains: List[str] = ["research", "tutoring"]):
    """
    Process all timelines in the mix directory and insert queries.
    """
    mix_dir = Path(mix_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    stats = {
        "total_timelines": 0,
        "processed_timelines": 0,
        "total_queries_inserted": 0,
        "by_domain": defaultdict(lambda: {"timelines": 0, "queries": 0}),
        "by_task": defaultdict(lambda: {"queries": 0})
    }
    
    for domain in domains:
        domain_dir = mix_dir / domain
        if not domain_dir.exists():
            print(f"Warning: Domain directory {domain_dir} does not exist")
            continue
        
        domain_output_dir = output_dir / domain
        domain_output_dir.mkdir(parents=True, exist_ok=True)
        
        # Get all numbered subdirectories
        subdirs = [d for d in domain_dir.iterdir() if d.is_dir() and d.name.isdigit()]
        subdirs.sort(key=lambda x: int(x.name))
        
        print(f"\nProcessing {domain} domain: {len(subdirs)} timelines")
        
        for subdir in subdirs:
            directory_index = subdir.name
            stats["total_timelines"] += 1
            stats["by_domain"][domain]["timelines"] += 1
            
            try:
                # Load timeline data
                events, artifact_history, concepts = load_timeline_data(mix_dir, domain, directory_index)
                
                # Get query rules for this domain
                domain_query_rules = QUERY_RULES.get(domain, {})
                
                # Insert queries
                new_events = insert_queries_for_timeline(events, artifact_history, concepts, 
                                                        domain, domain_query_rules)
                
                # Count queries inserted
                queries_count = sum(len(e.get("query", [])) for e in new_events)
                stats["total_queries_inserted"] += queries_count
                stats["by_domain"][domain]["queries"] += queries_count
                
                # Count by task type
                for event in new_events:
                    for query_info in event.get("query", []):
                        task_type = query_info.get("task", "Unknown")
                        stats["by_task"][task_type]["queries"] += 1
                
                # Save updated events
                output_timeline_dir = domain_output_dir / directory_index
                output_timeline_dir.mkdir(parents=True, exist_ok=True)
                
                output_events_file = output_timeline_dir / "events.json"
                with open(output_events_file, 'w', encoding='utf-8') as f:
                    json.dump(new_events, f, indent=2, ensure_ascii=False)
                
                stats["processed_timelines"] += 1
                
                if stats["processed_timelines"] % 10 == 0:
                    print(f"  Processed {stats['processed_timelines']} timelines...")
                    
            except Exception as e:
                print(f"  Error processing {domain}/{directory_index}: {e}")
                continue
    
    # Save statistics
    stats_file = output_dir / "insert_query_stats.json"
    # Convert defaultdict to regular dict for JSON serialization
    stats_serializable = {
        "total_timelines": stats["total_timelines"],
        "processed_timelines": stats["processed_timelines"],
        "total_queries_inserted": stats["total_queries_inserted"],
        "by_domain": dict(stats["by_domain"]),
        "by_task": {k: dict(v) for k, v in stats["by_task"].items()}
    }
    with open(stats_file, 'w', encoding='utf-8') as f:
        json.dump(stats_serializable, f, indent=2, ensure_ascii=False)
    
    print(f"\n=== Summary ===")
    print(f"Total timelines: {stats['total_timelines']}")
    print(f"Processed timelines: {stats['processed_timelines']}")
    print(f"Total queries inserted: {stats['total_queries_inserted']}")
    print(f"\nBy domain:")
    for domain, domain_stats in stats["by_domain"].items():
        print(f"  {domain}: {domain_stats['timelines']} timelines, {domain_stats['queries']} queries")
    print(f"\nBy task type:")
    for task_type, task_stats in stats["by_task"].items():
        print(f"  {task_type}: {task_stats['queries']} queries")
    
    return stats


In [ ]:
# Main execution
ALL_CONTEXT_MERGE_DIR = Path(".").resolve()
if not (ALL_CONTEXT_MERGE_DIR / "query_rules.py").exists():
    ALL_CONTEXT_MERGE_DIR = Path("data_pipeline/context_merge").resolve()
FRAMEWORK_DIR = ALL_CONTEXT_MERGE_DIR.parent

mix_dir = FRAMEWORK_DIR / "scenario_assembly"
output_dir = ALL_CONTEXT_MERGE_DIR / "output"

# Process all timelines
stats = process_all_timelines(mix_dir, output_dir, domains=["research", "tutoring"])